#### Report Subject
This report presents an analysis of medical data from patients with a history of oncological diseases. It was developed to investigate the risk factors and causes of cancer occurrence, as well as to evaluate the effectiveness of various treatment methods.

#### Data Source

https://www.kaggle.com/datasets/preetigupta004/cancer-issue

The data is sourced from Kaggle, a public dataset repository. This dataset contains cancer patient records collected from various regions, covering demographic details, lifestyle factors, cancer diagnostics, treatment information, and outcomes for 17,686 patients. The data is structured to facilitate the analysis of cancer diagnosis patterns and the effectiveness of medical treatments.


In [0]:
d = spark.read.format("csv").option("header", "true").load("/Workspace/Shared/cancer_issue_data.csv")

In [0]:
# d.printSchema()

In [0]:
import pyspark.sql.functions as f

In [0]:
# display(d)

#### Data Transformation and Analysis

##### Changing data type to "float" in variables: Age, BMI, TumorSize, SurvivalMonths

In [0]:
# d.printSchema()

In [0]:
d = d.withColumn("Age", d["Age"].cast("float")) \
     .withColumn("BMI", d["BMI"].cast("float")) \
     .withColumn("TumorSize", d["TumorSize"].cast("float")) \
     .withColumn("SurvivalMonths", d["SurvivalMonths"].cast("float"))

In [0]:
# d.printSchema()

##### Adding columns to a dataframe

###### Age groups
- 0-25 years old - Youth
- 25-60 years old - Adults
- ponad 60 years old - Seniors


In [0]:
d = d.withColumn("AgeGroup",
                   f.when(d.Age<25, "Youth").when((d.Age>=25) & (d.Age<60), "Adults").otherwise("Seniors"))

###### BMI Category
- <16,0	Severe underweight
- 16,0 – 16,9	Moderate underweight
- 17,0 - 18,5	Mild underweight
- 18,5 – 24,9	Normal weight
- 25,0 – 29,9	Overweight
- 30,0 – 34,9	Obesity Class I
- 35,0 – 39,9	Obesity Class II
- ≥40	Obesity Class III

In [0]:
d = d.withColumn("BMICategory",
                 f.when(d.BMI < 16.0, "Severe underweight")
                 .when((d.BMI >= 16.0) & (d.BMI < 17.0), "Moderate underweight")
                 .when((d.BMI >= 17.0) & (d.BMI < 18.5), "Mild underweight")
                 .when((d.BMI >= 18.5) & (d.BMI < 25.0), "Normal weight")
                 .when((d.BMI >= 25.0) & (d.BMI < 30.0), "Overweight")
                 .when((d.BMI >= 30.0) & (d.BMI < 35.0), "Obesity Class I")
                 .when((d.BMI >= 35.0) & (d.BMI < 40.0), "Obesity Class II")
                 .otherwise("Obesity Class III"))

In [0]:
# display(d)

##### Analysis of factors influencing the occurrence of cancer:
- Smoking status
- BMI
- Family history
- Age

###### Smoking status

In [0]:
d.groupBy("SmokingStatus").agg(
    f.count("CancerType").alias("Case_Count")
).display()

SmokingStatus,Case_Count
Former Smoker,5915
Smoker,5860
Non-Smoker,5911


Databricks visualization. Run in Databricks to view.

###### BMI

In [0]:
d.groupBy("BMICategory").agg(
    f.count("CancerType").alias("Case_Count")
).display()

BMICategory,Case_Count
Normal weight,5286
Obesity Class II,4107
Obesity Class I,4024
Overweight,4234
Obesity Class III,35


Databricks visualization. Run in Databricks to view.

###### Family medical history

In [0]:
d.groupBy("FamilyHistory").agg(
    f.count("CancerType").alias("Case_Count")
).display()

FamilyHistory,Case_Count
Yes,8863
No,8823


###### Age

In [0]:
d.groupBy("AgeGroup").agg(
    f.count("CancerType").alias("Case_Count")
).display()

AgeGroup,Case_Count
Youth,1683
Seniors,7447
Adults,8556


Databricks visualization. Run in Databricks to view.

##### The impact of smoking status on the occurrence of other types of cancer than lung cancer (without correlating variables):
- no group of people too young to be classified as smokers
- people with no family history of cancer
- people with a BMI considered normal (with a small deviation)
- people who do not have lung cancer


In [0]:
d_smoking_factor = d.filter(
    (d.Age >= 15) & 
    (d.FamilyHistory == "No") & 
    (d.BMI >= 17) & 
    (d.BMI < 32)&
    (d.CancerType != "Lung"))

In [0]:
d_smoking_factor.groupBy("SmokingStatus").agg(
    f.count("CancerType").alias("Case_Count")
).display()

SmokingStatus,Case_Count
Former Smoker,1543
Non-Smoker,1527
Smoker,1528


##### Analysis of the effectiveness of treatment using specific methods

In [0]:
d_treatment_success = d.groupBy("TreatmentType").agg(
    f.count("*").alias("Total_Treatments"),
    f.sum(f.when(f.col("TreatmentResponse") == "Complete Remission", 1).otherwise(0)).alias("Successful_Treatments"),
    f.round((f.col("Successful_Treatments") / f.col("Total_Treatments")) * 100, 2).alias
    ("Success_Rate_Percentage"))

In [0]:
d_treatment_success.orderBy(f.col("Success_Rate_Percentage").desc()).display()

TreatmentType,Total_Treatments,Successful_Treatments,Success_Rate_Percentage
Radiation,4272,1413,33.08
Surgery,4493,1483,33.01
Chemotherapy,4473,1472,32.91
Combination Therapy,4448,1425,32.04


Databricks visualization. Run in Databricks to view.

###### Saving the results of treatment effectiveness analysis to Hive

In [0]:
d_treatment_success.createOrReplaceTempView("tmp_treatment_success_rate")

In [0]:
%sql
create or replace table perm_treatment_success_rate as select * from tmp_treatment_success_rate

num_affected_rows,num_inserted_rows


In [0]:
%sql
show tables

database,tableName,isTemporary
default,perm_treatment_success_rate,false
,tmp_treatment_success_rate,true


In [0]:
%sql
select * from perm_treatment_success_rate

TreatmentType,Total_Treatments,Successful_Treatments,Success_Rate_Percentage
Combination Therapy,4448,1425,32.04
Radiation,4272,1413,33.08
Surgery,4493,1483,33.01
Chemotherapy,4473,1472,32.91


#### Summary and conclusions

The above analysis of medical data included an examination of diagnostic factors affecting cancer incidence, as well as an examination of the effectiveness of treatment using specific methods.

It is not possible to observe trends, as the data from the publicly available repository was generated randomly. Therefore, there is no correlation between the individual variables. Nevertheless, the report contains tables and visualisations in the form of graphs, which present the results of the analysis in an accessible way.